# Extending Jupyter Notebooks
In this notebook we will learn how we can use Python to modify cells in the Jupyter notebook we are executing right now using [IPython](https://ipython.org/).

In [1]:
import IPython

## Adding cells
First we setup a function that can add a new cell to our jupyter notebook.

In [2]:
def add_code_cell_below(code, replace_current_cell=False, execute_code:bool=False):
    """
    Add a new code cell to the currently running Jupyter Notebook.
    Optional: Replace the current cell instead of creating a new one.
    Optional: Execute the code
    """
    from IPython.core.getipython import get_ipython

    p = get_ipython()

    p.set_next_input(code, replace=replace_current_cell)
    if execute_code:
        p.run_cell(code)

In [3]:
add_code_cell_below("print('Hello World')")

In [ ]:
print('Hello World')

## Jupyter Magics
Next, we introduce a new Jupyter magic, that allows us to handle text in Jupyter cells when the user hits SHIFT+ENTER.

In [4]:
from IPython.core.magic import register_line_cell_magic
from llm_endpoints import prompt_ollama

@register_line_cell_magic
def alice(line: str, cell: str = ""):
    # ask LLM to write code
    code = prompt_ollama(f"""Please write Python code which does this: 
{line}
{cell}

Do not explain anything, just provide the code.""", model="gemma3:1b")
    
    # clean output
    code = code.strip("\n").strip("```python").strip("```").strip("\n")
    
    add_code_cell_below(code)

In [5]:
%alice print Hello world!

In [ ]:
print("Hello world!")

## Exercise
Create an image processing workflow by writing english prompts only. Try to not edit code manually.

In [8]:
%%alice please write a python function that can 
segment bright objects in an image and return a label image. 
Use scikit-image and numpy preferably.
Keep the code short and concise.

In [ ]:
import numpy as np
from skimage.feature import extract
from skimage.color import rgb2gray

def segment_bright_objects(image, threshold=127):
    """
    Segments bright objects in an image.

    Args:
        image (numpy.ndarray): The input image.
        threshold (int): The threshold for determining bright objects.

    Returns:
        numpy.ndarray: A label image with bright objects detected.
    """
    gray_image = rgb2gray(image)
    features = extract(gray_image, features=None)
    
    labels = np.zeros(gray_image.shape[:2], dtype=np.uint8)
    
    for i in range(gray_image.shape[0]):
        for j in range(gray_image.shape[1]):
            if features[i, j] > threshold:
                labels[i, j] = 1
    
    return labels

In [ ]:
import cv2
import numpy as np

def segment_bright_objects(image, threshold1=127, threshold2=144):
    """
    Segments bright objects in an image.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Apply thresholding
    _, thresholded = cv2.threshold(gray, threshold1, threshold2, cv2.THRESH_BINARY)

    # Find contours
    contours, _ = cv2.findContours(thresholded, cv2.cv.ARC)

    if contours:
        # Find the largest contour
        largest_contour = max(contours, key=cv2.contourArea)

        # Create a mask
        mask = np.zeros(image.shape[:2], dtype=np.uint8)

        # Iterate through the contours and use the contour as a mask
        for contour in contours:
            x, y, w, h = cv2.perimeterClip(contour, 11, 11, 11, 1, cv2.CHAIN_APPROX_SIMPLE)
            if w > 10 and h > 10: 
                mask = mask * 255 

        return mask
    else:
        return None

In [ ]:
def segment_image(image):
    """Segments an image into rectangular regions."""
    
    if not image:
        return []

    rows, cols = image.shape[:2]
    regions = []
    
    for i in range(rows):
        for j in range(cols):
            if i == 0 or i == rows - 1 or j == 0 or j == cols - 1:
                region = []
                for x in range(i, i + 1):
                    for y in range(j, j + 1):
                        region.append((x, y))
                regions.append(region)
    return regions